<a href="https://colab.research.google.com/github/zzzer0-wav/myDTA_2026/blob/main/classmade/demo_stat_tests(7).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 Статистичні тести в Python — демонстрація

**Тривалість:** ~25 хвилин · бібліотека `scipy.stats`

---

Це демонстраційний notebook до лекції про p-значення. Ми побачимо, що технічно кожен тест — це **лише кілька рядків коду**. Уся складність — у розумінні, *який* тест обрати і *як* прочитати результат.

**Нагадування з лекції:**
- `p < 0.05` → результат **статистично значущий** → відхиляємо H₀ (ефект є)
- `p ≥ 0.05` → **не значущий** → не відхиляємо H₀ (немає доказів ефекту)

> 💡 Усі тести повертають дві головні величини: **статистику тесту** і **p-значення**. Нас майже завжди цікавить саме p-значення.


## Підготовка

In [6]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv("shop_customers.csv")
print("Розмір:", df.shape)
df.head()

Розмір: (500, 11)


,customer_id,gender,age,country,channel,device,spend,session_min,sat_before,sat_after,purchased
0,1001,Ч,23,Німеччина,Реклама,Desktop,997.11,22.2,6,7,Так
1,1002,Ч,38,Україна,Органіка,Desktop,510.09,19.8,5,8,Ні
2,1003,Ч,20,Україна,Соцмережі,Desktop,789.71,21.0,7,8,Ні
3,1004,Ч,40,Німеччина,Соцмережі,Desktop,1041.02,17.4,8,9,Ні
4,1005,Ч,18,Україна,Реклама,Desktop,944.34,23.5,6,7,Ні


---
## Тест 1. Одновибірковий t-тест

**Питання:** чи середні витрати клієнтів відрізняються від очікуваних 900 грн?

**Гіпотези:**
- H₀: середні витрати = 900
- H₁: середні витрати ≠ 900

**Функція:** `stats.ttest_1samp(дані, очікуване_значення)`

In [7]:
mean_spend = df["spend"].mean()
print(f"Фактичне середнє: {mean_spend:.1f} грн")

t_stat, p_value = stats.ttest_1samp(df["spend"], 900)
print(f"t-статистика: {t_stat:.3f}")
print(f"p-значення:   {p_value:.4f}")

if p_value < 0.05:
    print("\n✅ p < 0.05 → значущо: середнє ВІДРІЗНЯЄТЬСЯ від 900")
else:
    print("\n❌ p ≥ 0.05 → не значущо: немає підстав казати, що середнє ≠ 900")

Фактичне середнє: 845.0 грн
t-статистика: -3.687
p-значення:   0.0003

✅ p < 0.05 → значущо: середнє ВІДРІЗНЯЄТЬСЯ від 900


**Інтерпретація:** p ≈ 0.0003 < 0.05 → відхиляємо H₀. Середні витрати (≈845 грн) статистично значущо відрізняються від 900 грн. Різниця не випадкова.

---
## Тест 2. Двовибірковий t-тест (незалежні групи)

**Питання:** чи відрізняються витрати чоловіків і жінок?

**Гіпотези:**
- H₀: середні витрати чоловіків = жінок
- H₁: вони відрізняються

**Функція:** `stats.ttest_ind(група1, група2)`

In [8]:
spend_m = df[df["gender"] == "Ч"]["spend"]
spend_f = df[df["gender"] == "Ж"]["spend"]

print(f"Чоловіки: {spend_m.mean():.1f} грн")
print(f"Жінки:    {spend_f.mean():.1f} грн")

t_stat, p_value = stats.ttest_ind(spend_m, spend_f)
print(f"\np-значення: {p_value:.4f}")

if p_value < 0.05:
    print("✅ Значуща різниця між статями")
else:
    print("❌ Значущої різниці НЕ виявлено")

Чоловіки: 848.7 грн
Жінки:    841.3 грн

p-значення: 0.8038
❌ Значущої різниці НЕ виявлено


**Інтерпретація:** p ≈ 0.80 > 0.05 → **не відхиляємо** H₀. Ми не виявили значущої різниці у витратах між чоловіками і жінками. Зверніть увагу: ми кажемо «не виявили», а не «довели, що різниці немає»!

---
## Тест 3. Парний t-тест (до і після)

**Питання:** чи зросла задоволеність клієнтів після редизайну сайту?

Тут ми порівнюємо ДВА виміри в ОДНИХ І ТИХ САМИХ клієнтів (до і після) — тому тест **парний**.

**Функція:** `stats.ttest_rel(до, після)`

In [9]:
print(f"Задоволеність ДО:    {df['sat_before'].mean():.2f}")
print(f"Задоволеність ПІСЛЯ: {df['sat_after'].mean():.2f}")

t_stat, p_value = stats.ttest_rel(df["sat_before"], df["sat_after"])
print(f"\np-значення: {p_value:.6f}")

if p_value < 0.05:
    print("✅ Задоволеність значущо змінилася")
else:
    print("❌ Значущої зміни не виявлено")

Задоволеність ДО:    6.54
Задоволеність ПІСЛЯ: 7.14

p-значення: 0.000000
✅ Задоволеність значущо змінилася


**Інтерпретація:** p < 0.001 → відхиляємо H₀. Задоволеність значущо зросла (з 6.54 до 7.14). Редизайн спрацював.

> 💡 Чому саме парний тест? Бо кожне «після» пов'язане зі своїм «до» (той самий клієнт). Парний тест враховує цей зв'язок і тому потужніший.

---
## Тест 4. ANOVA (порівняння 3+ груп)

**Питання:** чи відрізняються витрати залежно від каналу залучення (Органіка / Реклама / Соцмережі)?

Груп **три**, тому t-тест не підходить — використовуємо **ANOVA**.

**Функція:** `stats.f_oneway(група1, група2, група3)`

In [10]:
for channel in df["channel"].unique():
    avg = df[df["channel"] == channel]["spend"].mean()
    print(f"{channel}: {avg:.1f} грн")

groups = [df[df["channel"] == ch]["spend"] for ch in df["channel"].unique()]
f_stat, p_value = stats.f_oneway(*groups)
print(f"\nF-статистика: {f_stat:.2f}")
print(f"p-значення:   {p_value:.6f}")

if p_value < 0.05:
    print("✅ Принаймні один канал значущо відрізняється")
else:
    print("❌ Значущої різниці між каналами немає")

Реклама: 1057.9 грн
Органіка: 791.3 грн
Соцмережі: 698.4 грн

F-статистика: 63.54
p-значення:   0.000000
✅ Принаймні один канал значущо відрізняється


**Інтерпретація:** p < 0.001 → відхиляємо H₀. Канали значущо відрізняються за витратами (Реклама ≈1058 грн помітно вища за Соцмережі ≈698).

> ⚠️ ANOVA каже лише, що *якась* різниця є, але не каже, *між якими саме* каналами. Щоб дізнатись це, потрібні додаткові (post-hoc) тести — це вже за межами сьогоднішньої демонстрації.

---
## Тест 5. Хі-квадрат (звʼязок категорій)

**Питання:** чи пов'язаний тип пристрою (Mobile/Desktop) з фактом покупки (Так/Ні)?

Обидві змінні **категоріальні** → хі-квадрат.

**Крок 1:** будуємо таблицю спряженості. **Крок 2:** `stats.chi2_contingency(таблиця)`

In [11]:
contingency = pd.crosstab(df["device"], df["purchased"])
print("Таблиця спряженості:")
print(contingency)

chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
print(f"\nchi2: {chi2:.3f}")
print(f"p-значення: {p_value:.4f}")

if p_value < 0.05:
    print("✅ Пристрій ПОВ'ЯЗАНИЙ із покупкою")
else:
    print("❌ Зв'язку не виявлено")

Таблиця спряженості:
purchased   Ні  Так
device             
Desktop    110   79
Mobile     223   88

chi2: 9.039
p-значення: 0.0026
✅ Пристрій ПОВ'ЯЗАНИЙ із покупкою


**Інтерпретація:** p ≈ 0.003 < 0.05 → відхиляємо H₀. Тип пристрою пов'язаний з імовірністю покупки (на Desktop купують частіше). Це корисний інсайт для бізнесу!

---
## Тест 6. Кореляція (зв'язок двох чисел)

**Питання:** чи пов'язаний час на сайті із сумою витрат?

Обидві змінні **числові** → кореляція Пірсона.

**Функція:** `stats.pearsonr(змінна1, змінна2)` — повертає коефіцієнт r та p-значення.

In [12]:
r, p_value = stats.pearsonr(df["session_min"], df["spend"])
print(f"Коефіцієнт кореляції r: {r:.3f}")
print(f"p-значення: {p_value:.6f}")

# r близький до 1 → сильний прямий зв'язок; до 0 → зв'язку немає
if p_value < 0.05:
    print("✅ Зв'язок статистично значущий")
else:
    print("❌ Значущого зв'язку немає")

Коефіцієнт кореляції r: 0.732
p-значення: 0.000000
✅ Зв'язок статистично значущий


**Інтерпретація:** r ≈ 0.73 (сильний прямий зв'язок) і p < 0.001 (значущий). Чим більше часу клієнт на сайті, тим більше витрачає.

> ⚠️ **Кореляція ≠ причинність!** Можливо, час спричиняє витрати, можливо — навпаки, а можливо, є третій фактор (зацікавленість). Тест каже лише, що зв'язок реальний.

---
# Підсумок демонстрації

Ми провели **6 різних тестів**, і кожен — це буквально 1-2 рядки коду:

| Тест | Функція scipy | Коли застосовувати |
|------|---------------|-------------------|
| Одновибірковий t | `ttest_1samp` | середнє проти фіксованого числа |
| Двовибірковий t | `ttest_ind` | порівняти 2 незалежні групи |
| Парний t | `ttest_rel` | до/після в тих самих об'єктів |
| ANOVA | `f_oneway` | порівняти 3+ групи |
| Хі-квадрат | `chi2_contingency` | зв'язок 2 категорій |
| Кореляція | `pearsonr` | зв'язок 2 чисел |

**Головна думка:** код — це проста частина. Майстерність аналітика — у тому, щоб:
1. обрати **правильний** тест під тип даних;
2. правильно **прочитати** p-значення;
3. памʼятати про **контекст** (розмір ефекту, причинність, практична важливість).

Тепер — ваша черга практикуватись! 🚀
